# MSNet Training on Washington RGB-D Object - Google Colab

**Hyperparameter tuning with Ray Tune using the on-disk train/val split (from the preprocess pipeline).**

---

## Checklist Before Running:
- [ ] **Enable A100 GPU:** Runtime > Change runtime type > A100
- [ ] **Washington dataset on Drive:** `Shareddrives/MSNN-Capstone/data/washington_rgbd_256.tar.gz` (produced by `notebooks/washington_preprocess.ipynb`)

Training uses: `MSNet` (no pretraining, no backbone freezing — trained from random init on all trials).

## 1. Environment Setup & GPU Verification

In [ ]:
# Check GPU availability and specs
import torch
import subprocess

print("=" * 60)
print("GPU VERIFICATION")
print("=" * 60)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

    gpu_name = torch.cuda.get_device_name(0)
    if 'A100' in gpu_name:
        print("\nA100 GPU detected - optimal for training")
    elif 'V100' in gpu_name:
        print("\nV100 GPU detected - good for training (slower than A100)")
    elif 'T4' in gpu_name:
        print("\nT4 GPU detected - will be slower, consider upgrading to A100")
    else:
        print(f"\nGPU: {gpu_name}")
else:
    print("\nNO GPU DETECTED!")
    print("Enable GPU: Runtime -> Change runtime type -> Hardware accelerator: GPU")
    raise RuntimeError("GPU is required for training")

print("\n" + "=" * 60)

In [ ]:
# Detailed GPU info
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

print("\nGoogle Drive mounted successfully!")
print(f"\nShared drive contents:")
!ls -la /content/drive/Shareddrives/MSNN-Capstone/ | head -20

## 3. Clone Repository to Local Disk (Fast I/O)

**Important:** We clone to `/content/` (local SSD) instead of Drive for 10-20x faster I/O

In [ ]:
import os
from pathlib import Path

PROJECT_NAME = "MSNN-Capstone"
GITHUB_REPO = "https://github.com/clingergab/MSNN-Capstone.git"
LOCAL_REPO_PATH = f"/content/{PROJECT_NAME}"

print("=" * 60)
print("REPOSITORY SETUP")
print("=" * 60)

os.chdir('/content')

if Path(LOCAL_REPO_PATH).exists() and Path(f"{LOCAL_REPO_PATH}/.git").exists():
    print(f"Repo already exists: {LOCAL_REPO_PATH}")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
else:
    if Path(LOCAL_REPO_PATH).exists():
        !rm -rf {LOCAL_REPO_PATH}
    print(f"Cloning from {GITHUB_REPO}...")
    !git clone {GITHUB_REPO} {LOCAL_REPO_PATH}
    if not Path(LOCAL_REPO_PATH).exists():
        raise RuntimeError(f"Failed to clone repository")
    os.chdir(LOCAL_REPO_PATH)

print(f"\nWorking directory: {os.getcwd()}")
!ls -la {LOCAL_REPO_PATH}
print("\n" + "=" * 60)

## 4. Install Dependencies

In [ ]:
print("Installing dependencies...")

!pip install -q h5py tqdm matplotlib seaborn ray[tune] optuna kornia thop

import h5py
import tqdm
import matplotlib
import seaborn
import ray
import kornia

print("All dependencies installed!")
print(f"   h5py: {h5py.__version__}")
print(f"   matplotlib: {matplotlib.__version__}")
print(f"   ray: {ray.__version__}")
print(f"   kornia: {kornia.__version__}")

## 5. Copy Washington RGB-D Dataset to Local Disk

**Performance Note:** Local disk (`/dev/shm`) is ~10-20x faster than Drive!

**Dataset:** Washington RGB-D Object 51-category preprocessed (train + val + test splits, paired `.pt` tensors)

In [ ]:
from pathlib import Path
import os

DRIVE_DATASET_TAR = "/content/drive/Shareddrives/MSNN-Capstone/data/washington_rgbd_256.tar.gz"
LOCAL_DATASET_PATH = "/dev/shm/washington_rgbd_256"

print("=" * 60)
print("WASHINGTON RGB-D OBJECT DATASET SETUP (51 CATEGORIES)")
print("=" * 60)

if Path(LOCAL_DATASET_PATH).exists():
    print(f"Already on local disk: {LOCAL_DATASET_PATH}")
    for split in ['train', 'val', 'test']:
        split_dir = Path(f"{LOCAL_DATASET_PATH}/{split}")
        if split_dir.exists():
            n = len(list(split_dir.glob('*/*_rgb.pt')))
            print(f"   {split.capitalize()} samples: {n}")
elif Path(DRIVE_DATASET_TAR).exists():
    print(f"Found on Drive: {DRIVE_DATASET_TAR}")
    tar_name = Path(DRIVE_DATASET_TAR).name
    local_tar = f"/dev/shm/{tar_name}"
    !rsync -ah --info=progress2 "{DRIVE_DATASET_TAR}" {local_tar}
    print(f"\nExtracting...")
    !tar -xzf {local_tar} -C /dev/shm/ 2>&1 | grep -v "Ignoring unknown extended header"
    !rm {local_tar}
    for split in ['train', 'val', 'test']:
        split_dir = Path(f"{LOCAL_DATASET_PATH}/{split}")
        if split_dir.exists():
            n = len(list(split_dir.glob('*/*_rgb.pt')))
            print(f"   {split.capitalize()} samples: {n}")
else:
    raise FileNotFoundError(f"Dataset not found at {DRIVE_DATASET_TAR}")

for meta in ['class_names.txt', 'norm_stats.json']:
    meta_path = Path(LOCAL_DATASET_PATH) / meta
    status = 'OK' if meta_path.exists() else 'MISSING'
    print(f"   {meta}: {status}")

print(f"\nDataset ready at: {LOCAL_DATASET_PATH}")

## 6. Setup Python Path & Import MSNet

In [ ]:
import sys
import os

modules_to_reload = [k for k in sys.modules.keys() if k.startswith('src.')]
for module in modules_to_reload:
    del sys.modules[module]

project_root = '/content/MSNN-Capstone'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Project structure:")
!ls -la {project_root}/src/models/

print("\nImporting MSNet and dataloaders...")
from src.models.linear_integration.ms_net import ms_resnet18
from src.data_utils.washington_dataset import (
    WashingtonRGBDDataset,
    get_washington_dataloaders,
    _load_class_names,
    _load_norm_stats,
    _discover_samples,
)
from src.training.augmentation_config import AugmentationConfig

from ray import train, tune
from ray.tune.schedulers import ASHAScheduler

print("All imports successful!")

## 7. Hyperparameter Tuning with Ray Tune

- **Parallel Trials:** Run multiple configurations simultaneously
- **Train/Val Split:** Uses the on-disk `train/` and `val/` directories produced by the preprocess pipeline
- **ASHA Scheduler:** Early-stop unpromising trials
- **Resumable:** Experiment state saved to Google Drive, survives Colab restarts
- **Random Init Only:** No pretraining or backbone freezing - every trial starts from scratch

In [ ]:
import os
import time

# 1. Define Paths explicitly
mps_pipe_dir = "/tmp/nvidia-mps"
mps_log_dir = "/tmp/nvidia-log"

# 2. Create the directories (CRITICAL: Daemon fails if log dir doesn't exist)
os.makedirs(mps_pipe_dir, exist_ok=True)
os.makedirs(mps_log_dir, exist_ok=True)

# 3. Set Environment Variables for the current Python process
os.environ["CUDA_MPS_PIPE_DIRECTORY"] = mps_pipe_dir
os.environ["CUDA_MPS_LOG_DIRECTORY"] = mps_log_dir
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

# 4. Configure GPU and Start Daemon using the SAME environment variables
print("Setting GPU to Exclusive Process Mode...")
!nvidia-smi -i 0 -c EXCLUSIVE_PROCESS

print("Starting MPS Daemon...")
!export CUDA_MPS_PIPE_DIRECTORY={mps_pipe_dir} && \
 export CUDA_MPS_LOG_DIRECTORY={mps_log_dir} && \
 nvidia-cuda-mps-control -d

# 5. Verify it is running
print("Verifying Daemon Status...")
time.sleep(1)
!ps -ef | grep mps

if os.path.exists(os.path.join(mps_pipe_dir, "control")):
    print("MPS Control Pipe found. Setup success.")
else:
    print("MPS Control Pipe NOT found. Check /tmp/nvidia-log for errors.")
    !cat {mps_log_dir}/control.log

In [ ]:
import random
import numpy as np

import ray
from ray import tune
from ray.tune.schedulers import ASHAScheduler
from ray.tune.search.optuna import OptunaSearch
from optuna.samplers import TPESampler
import torch
from collections import Counter

from src.models.linear_integration.ms_net import ms_resnet18
from src.training.optimizers import create_stream_optimizer
from src.training.schedulers import setup_scheduler
from src.data_utils.washington_dataset import (
    WashingtonRGBDDataset,
    _load_norm_stats,
    _load_class_names,
    _discover_samples,
)
from src.training.augmentation_config import AugmentationConfig
from src.utils.seed import set_seed


class TrialTerminated(Exception):
    """Raised when a trial should be terminated early."""
    pass


class RayTuneReporter:
    """Callback for reporting metrics to Ray Tune during training."""

    def __init__(self):
        self.best_accuracy = 0.0
        self.best_loss = float('inf')
        self.best_train_acc = 0.0
        self.best_val_mca = 0.0
        self.best_train_mca = 0.0
        self.best_epoch = 0

    def on_epoch_end(self, epoch, logs):
        """Report current AND best metrics to Ray Tune."""
        if logs['val_accuracy'] > self.best_accuracy:
            self.best_accuracy = logs['val_accuracy']
            if logs['train_accuracy'] > self.best_train_acc:
                self.best_train_acc = logs['train_accuracy']

        if logs['val_loss'] < self.best_loss:
            self.best_loss = logs['val_loss']

        val_mca = logs.get('val_mca', 0.0)
        train_mca = logs.get('train_mca', 0.0)
        if val_mca > self.best_val_mca:
            self.best_val_mca = val_mca
            self.best_epoch = epoch
            if train_mca > self.best_train_mca:
                self.best_train_mca = train_mca

        gap = self.best_train_mca - self.best_val_mca
        composite = self.best_val_mca - 5 * (gap**3)

        tune.report({
            "accuracy": logs['val_accuracy'],
            "loss": logs['val_loss'],
            "best_accuracy": self.best_accuracy,
            "best_loss": self.best_loss,
            "train_loss": logs['train_loss'],
            "train_accuracy": logs['train_accuracy'],
            "best_train_acc": self.best_train_acc,
            "val_mca": val_mca,
            "train_mca": train_mca,
            "best_val_mca": self.best_val_mca,
            "best_train_mca": self.best_train_mca,
            "gap": gap,
            "best_epoch": self.best_epoch,
        })


def train_msnet_tune(
    config,
    data_root=None,
    norm_stats=None,
    seed=42,
    class_names=None,
    train_samples=None,
    val_samples=None,
):
    """
    Trainable function for Ray Tune. Trains MSNet on Washington RGB-D Object
    from random init (no pretraining, no backbone freezing).

    Args:
        config: Ray Tune configuration dict with hyperparameters
        data_root: Path to dataset root (with train/ and val/ directories)
        norm_stats: Normalization statistics dict
        seed: Random seed for reproducible trials
        class_names: Class name list (pre-loaded from class_names.txt)
        train_samples: Pre-discovered (rgb_path, depth_path, label) tuples for train/
        val_samples: Pre-discovered (rgb_path, depth_path, label) tuples for val/
    """
    set_seed(seed, deterministic=False)
    g = torch.Generator().manual_seed(seed)

    # Per-trial augmentation config
    aug_config = AugmentationConfig(
        rgb_aug_prob=config.get("rgb_aug_prob"),
        rgb_aug_mag=config.get("rgb_aug_mag"),
        depth_aug_prob=config.get("depth_aug_prob"),
        depth_aug_mag=config.get("depth_aug_mag"),
    )

    # Dataset: normalize=False because MSNet does GPU-side normalization via
    # model.compile(gpu_augmentation=True, norm_stats=norm_stats).
    train_dataset = WashingtonRGBDDataset(
        data_root=data_root,
        split='train',
        samples=train_samples,
        class_names=class_names,
        norm_stats=norm_stats,
        crop_size=224,
        normalize=False,
        **aug_config.to_dict(),
    )
    val_dataset = WashingtonRGBDDataset(
        data_root=data_root,
        split='val',
        samples=val_samples,
        class_names=class_names,
        norm_stats=norm_stats,
        crop_size=224,
        normalize=False,
    )

    # Weighted sampler for class-balanced training (inverse frequency)
    sample_weights = train_dataset.get_sample_weights()
    train_sampler = torch.utils.data.WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
        generator=g,
    )

    def worker_init_fn(worker_id):
        worker_seed = seed + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)
        torch.manual_seed(worker_seed)

    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=64,
        shuffle=False,
        sampler=train_sampler,
        num_workers=2,
        prefetch_factor=2,
        persistent_workers=True,
        pin_memory=True,
        worker_init_fn=worker_init_fn,
    )
    val_loader = torch.utils.data.DataLoader(
        val_dataset,
        batch_size=64,
        shuffle=False,
        num_workers=1,
        prefetch_factor=2,
        persistent_workers=False,
        pin_memory=True,
        worker_init_fn=worker_init_fn,
    )

    # Create MSNet (random init)
    model = ms_resnet18(
        num_classes=51,
        stream_input_channels=[3, 1],
        dropout_p=config["dropout_p"],
        width_multiplier=0.75,
        device="cuda",
        use_amp=True,
    )

    # Create Optimizer
    optimizer = create_stream_optimizer(
        model,
        optimizer_type='adamw',
        stream_lrs=[config["lr"], config["lr"]],
        stream_weight_decays=[config["wd"], config["wd"]],
        shared_lr=config["lr"],
        integration_weight_decay=config["wd"],
        stem_lr_multiplier=config["stem_lr_multiplier"],
    )

    # Create Scheduler (cosine with linear warmup)
    warmup_epochs = 5
    scheduler = setup_scheduler(
        optimizer,
        scheduler_type='cosine',
        # eta_min: 6 groups when stem split, 4 when not
        eta_min=(
            [config['eta_min'] * config['stem_lr_multiplier'],
             config['eta_min'] * config['stem_lr_multiplier'],
             config['eta_min'], config['eta_min'],
             config['eta_min'], config['eta_min']]
            if config['stem_lr_multiplier'] != 1.0 else
            [config['eta_min']] * 4
        ),
        t_max=45,
        train_loader_len=len(train_loader),
        warmup_epochs=warmup_epochs,
        warmup_start_factor=0.2,
    )

    # Compile (gpu_augmentation=True enables GPU-side normalization + appearance aug)
    model.compile(
        optimizer=optimizer,
        scheduler=scheduler,
        loss='cross_entropy',
        label_smoothing=config["label_smoothing"],
        gpu_augmentation=True,
        norm_stats=norm_stats,
        **aug_config.to_dict(),
    )

    # Train
    try:
        model.fit(
            train_loader=train_loader,
            val_loader=val_loader,
            epochs=50,
            early_stopping=True,
            monitor='val_mca',
            patience=15,
            grad_clip_norm=config["grad_clip_norm"],
            callbacks=[RayTuneReporter()],
            verbose=False,
        )
    except TrialTerminated as e:
        print(f"\n{e}")

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================
# Ray Tune saves ALL experiment state to DRIVE_STORAGE_PATH via storage_path.
# When Colab dies, re-run the notebook - Tuner.restore() picks up where it
# left off. Completed trials preserved, interrupted trials restart.
# =============================================================================

import os
import pandas as pd
from pathlib import Path

# --- Ray Tune persistent storage on Google Drive ---
DRIVE_STORAGE_PATH = "/content/drive/Shareddrives/MSNN-Capstone/ray_tune_experiments"
LOCAL_STORAGE_PATH = "/content/ray_results"
EXPERIMENT_NAME = "washington_rgbd_hpo"

SEED = 152
NUM_SAMPLES = 200  # Total trials to run across all sessions

Path(DRIVE_STORAGE_PATH).mkdir(parents=True, exist_ok=True)
Path(LOCAL_STORAGE_PATH).mkdir(parents=True, exist_ok=True)

experiment_path = os.path.join(DRIVE_STORAGE_PATH, EXPERIMENT_NAME)
local_experiment_path = os.path.join(LOCAL_STORAGE_PATH, EXPERIMENT_NAME)
RESUME_EXISTING = os.path.exists(experiment_path)

if RESUME_EXISTING:
    _exp_files = os.listdir(experiment_path) if os.path.isdir(experiment_path) else []
    if len(_exp_files) == 0:
        print(f"  WARNING: {experiment_path} exists but is empty - starting fresh")
        RESUME_EXISTING = False

print(f"Drive storage: {DRIVE_STORAGE_PATH}")
print(f"Local storage: {LOCAL_STORAGE_PATH}")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Resume existing: {RESUME_EXISTING}")
print(f"Total trials: {NUM_SAMPLES}")
print(f"Training: random-init MSNet (no pretraining, no freezing)")

if RESUME_EXISTING:
    print(f"\n  Previous experiment found at {experiment_path}")
    print(f"  Copying Drive -> local, then Tuner.restore() from local.")
    import subprocess as _sp_cfg
    os.makedirs(local_experiment_path, exist_ok=True)
    _result = _sp_cfg.run(
        ["rsync", "-a", experiment_path + "/", local_experiment_path + "/"],
        capture_output=True, text=True,
    )
    if _result.returncode == 0:
        print(f"  Restored to {local_experiment_path}")
    else:
        raise RuntimeError(f"Restore failed: {_result.stderr[:300]}")

In [ ]:
# Initialize Ray
import shutil
import subprocess
import time as _time

from ray.tune import CLIReporter
from ray.tune import Callback as TuneCallback

os.environ["RAY_AIR_NEW_OUTPUT"] = "0"  # must be set BEFORE ray.init()

class DriveSyncCallback(TuneCallback):
    """Periodically rsyncs local Ray Tune experiment state to Google Drive.

    Ray Tune writes to LOCAL_STORAGE_PATH (fast local disk).  This callback
    rsyncs local -> Drive incrementally (only changed files) so that state
    survives Colab session death without blocking the Ray driver.
    """

    def __init__(self, local_storage_path, drive_storage_path, experiment_name,
                 sync_interval_seconds=300):
        self._local_path = os.path.join(local_storage_path, experiment_name)
        self._drive_path = os.path.join(drive_storage_path, experiment_name)
        self._sync_interval = sync_interval_seconds
        self._last_sync = 0.0

    def _sync(self, reason=""):
        if not os.path.isdir(self._local_path):
            return
        try:
            os.makedirs(self._drive_path, exist_ok=True)
            result = subprocess.run(
                ["rsync", "-a", self._local_path + "/", self._drive_path + "/"],
                capture_output=True, text=True, timeout=120,
            )
            if result.returncode == 0:
                self._last_sync = _time.time()
                print(f"[DriveSyncCallback] synced to Drive ({reason})")
            else:
                print(f"[DriveSyncCallback] WARNING: rsync failed: {result.stderr[:200]}")
        except subprocess.TimeoutExpired:
            print(f"[DriveSyncCallback] WARNING: rsync timed out (120s)")
        except Exception as e:
            print(f"[DriveSyncCallback] WARNING: sync failed: {e}")

    def on_trial_result(self, iteration, trials, trial, result, **info):
        if _time.time() - self._last_sync >= self._sync_interval:
            self._sync(reason=f"periodic, iter={result.get('training_iteration', '?')}")

    def on_trial_complete(self, iteration, trials, trial, **info):
        if _time.time() - self._last_sync >= 60:
            self._sync(reason="trial complete")

    def on_experiment_end(self, trials, **info):
        self._sync(reason="experiment end")


class BestTrialReporter(TuneCallback):
    """Periodically prints the best trial's config and metrics."""

    def __init__(self, metric="best_val_mca", mode="max", every_n_results=20):
        self._metric = metric
        self._mode = mode
        self._every_n = every_n_results
        self._result_count = 0
        self._best_value = float('-inf') if mode == "max" else float('inf')
        self._best_config = None

    def on_trial_result(self, iteration, trials, trial, result, **info):
        self._result_count += 1
        val = result.get(self._metric, None)
        if val is None:
            return
        improved = (val > self._best_value) if self._mode == "max" else (val < self._best_value)
        if improved:
            self._best_value = val
            self._best_config = trial.config.copy()

        if self._result_count % self._every_n == 0 and self._best_config is not None:
            self._print_best(result)

    def _print_best(self, latest_result):
        print(f"\n{'-'*60}")
        print(f"  Best {self._metric}: {self._best_value*100:.2f}% "
              f"(after {self._result_count} results)")
        for k, v in self._best_config.items():
            if isinstance(v, float):
                print(f"    {k}: {v:.2e}" if abs(v) < 0.01 else f"    {k}: {v:.4f}")
            else:
                print(f"    {k}: {v}")
        print(f"{'-'*60}")


ray.shutdown()
ray.init(
    ignore_reinit_error=True,
    runtime_env={
        "env_vars": {
            "CUDA_MPS_PIPE_DIRECTORY": "/tmp/nvidia-mps",
            "CUDA_MPS_LOG_DIRECTORY": "/tmp/nvidia-log",
            "CUDA_DEVICE_ORDER": "PCI_BUS_ID",
            "CUDA_VISIBLE_DEVICES": "0",
        }
    }
)

# Load metadata + discover samples ONCE (passed to every trial via tune.with_parameters)
norm_stats = _load_norm_stats(LOCAL_DATASET_PATH)
class_names = _load_class_names(LOCAL_DATASET_PATH)
train_samples = _discover_samples(os.path.join(LOCAL_DATASET_PATH, 'train'), class_names)
val_samples = _discover_samples(os.path.join(LOCAL_DATASET_PATH, 'val'), class_names)
print(f"Dataset: {LOCAL_DATASET_PATH}")
print(f"  Classes: {len(class_names)}")
print(f"  Train samples: {len(train_samples)}")
print(f"  Val samples:   {len(val_samples)}")


# Define trainable (same for both new and restored runs)
trainable = tune.with_resources(
    tune.with_parameters(
        train_msnet_tune,
        data_root=LOCAL_DATASET_PATH,
        norm_stats=norm_stats,
        seed=SEED,
        class_names=class_names,
        train_samples=train_samples,
        val_samples=val_samples,
    ),
    resources={"cpu": 2, "gpu": 1.0 / 5},
)


# Callback to force-sync experiment state to Drive
drive_sync_cb = DriveSyncCallback(LOCAL_STORAGE_PATH, DRIVE_STORAGE_PATH, EXPERIMENT_NAME)


if RESUME_EXISTING:
    # =========================================================
    # RESUME: Restore previous experiment from Google Drive
    # =========================================================
    print("\n" + "=" * 60)
    print("RESUMING EXPERIMENT FROM GOOGLE DRIVE")
    print("=" * 60)

    tuner = tune.Tuner.restore(
        path=local_experiment_path,
        trainable=trainable,
        resume_unfinished=True,
        resume_errored=True,
    )

else:
    # =========================================================
    # NEW: Create fresh experiment
    # =========================================================
    print("\n" + "=" * 60)
    print("STARTING NEW EXPERIMENT")
    print("=" * 60)

    # Search space (Washington RGB-D Object tuned ranges)
    search_space = {
        # Learning rates
        "lr": tune.loguniform(5e-5, 3e-4),
        "wd": tune.loguniform(1e-4, 1e-2),

        # Scheduler eta_min
        "eta_min": tune.loguniform(1.0e-6, 3.0e-5),

        # Regularization
        "dropout_p": tune.quniform(0.40, 0.60, 0.01),
        "label_smoothing": tune.quniform(0.05, 0.15, 0.01),
        "grad_clip_norm": tune.quniform(0.8, 1.8, 0.05),

        # Stem learning rate multiplier (DLR for conv1)
        "stem_lr_multiplier": tune.quniform(10.0, 25.0, 1.0),

        # Augmentation parameters
        "rgb_aug_prob": tune.quniform(0.8, 1.2, 0.01),
        "rgb_aug_mag": tune.quniform(0.8, 1.2, 0.01),
        "depth_aug_prob": tune.quniform(0.8, 1.2, 0.01),
        "depth_aug_mag": tune.quniform(0.8, 1.2, 0.01),
    }

    reporter = CLIReporter(
        parameter_columns=[
            "lr",
            "wd",
            "eta_min",
            "dropout_p",
            "label_smoothing",
            "grad_clip_norm",
            "rgb_aug_prob",
            "rgb_aug_mag",
            "depth_aug_prob",
            "depth_aug_mag",
        ],
        metric_columns={
            "training_iteration": "iter",
            "best_val_mca": "best_val_mca",
            "best_train_mca": "best_train_mca",
            "best_accuracy": "best_accuracy",
            "gap": "gap",
            "best_epoch": "best_epoch",
        },
        max_report_frequency=30,
        print_intermediate_tables=True,
    )

    optuna_search = OptunaSearch(
        metric="best_val_mca",
        mode="max",
        sampler=TPESampler(n_startup_trials=15),
    )

    asha_scheduler = ASHAScheduler(
        time_attr="training_iteration",
        metric="best_val_mca",
        mode="max",
        max_t=50,
        grace_period=15,
        reduction_factor=2,
    )

    tuner = tune.Tuner(
        trainable,
        param_space=search_space,
        tune_config=tune.TuneConfig(
            scheduler=asha_scheduler,
            search_alg=optuna_search,
            num_samples=NUM_SAMPLES,
            max_concurrent_trials=10,
        ),
        run_config=ray.tune.RunConfig(
            storage_path=LOCAL_STORAGE_PATH,
            name=EXPERIMENT_NAME,
            progress_reporter=reporter,
            verbose=1,
            callbacks=[drive_sync_cb, BestTrialReporter(every_n_results=20)],
        ),
    )


# Run (or resume) tuning
print("\n" + "=" * 60)
print("STARTING HYPERPARAMETER TUNING")
print("=" * 60)

results = tuner.fit()

best_result = results.get_best_result("best_val_mca", "max")

print("\n" + "=" * 60)
print("TUNING COMPLETE")
print("=" * 60)
print(f"Best Trial Config: {best_result.config}")
print(f"Best Trial Val MCA: {best_result.metrics['best_val_mca']:.4f}")
print(f"Best Trial Accuracy: {best_result.metrics['best_accuracy']:.4f}")
print(f"Best Trial Loss: {best_result.metrics['best_loss']:.4f}")
print(f"\nExperiment saved to: {local_experiment_path}")
print(f"Drive backup: {experiment_path}")
print(f"To resume after Colab dies: just re-run this notebook.")

In [ ]:
# =============================================================================
# SAVE RESULTS CSV (for offline analysis)
# =============================================================================
import datetime

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

results_df = results.get_dataframe()

csv_dir = f"{DRIVE_STORAGE_PATH}/analysis"
Path(csv_dir).mkdir(parents=True, exist_ok=True)

csv_path = f"{csv_dir}/washington_hpo_results_{timestamp}.csv"
results_df.to_csv(csv_path, index=False)
print(f"Results CSV saved: {csv_path}")
print(f"  Trials: {len(results_df)}")

latest_path = f"{csv_dir}/washington_hpo_results_latest.csv"
results_df.to_csv(latest_path, index=False)
print(f"Latest copy: {latest_path}")

In [ ]:
# Analyze Top 10 Trials from Ray Tune (ranked by best_val_mca)
import pandas as pd

print("=" * 80)
print("TOP 10 TRIALS BY BEST VAL MCA (CONTINUOUS SEARCH)")
print("=" * 80)

df = results.get_dataframe()
df_sorted = df.sort_values('gap', ascending=True)

display_cols = [
    'best_val_mca', 'best_train_mca', 'best_accuracy', 'best_train_acc', 'gap',
    'config/lr',
    'config/wd',
    'config/eta_min',
    'config/dropout_p',
    'config/label_smoothing',
    'config/grad_clip_norm',
    'config/rgb_aug_prob',
    'config/rgb_aug_mag',
    'config/depth_aug_prob',
    'config/depth_aug_mag',
    'config/stem_lr_multiplier',
]

top_10 = df_sorted[display_cols].head(30)

top_10_formatted = top_10.copy()
top_10_formatted['best_val_mca'] = top_10_formatted['best_val_mca'].apply(lambda x: f"{x*100:.2f}%")
top_10_formatted['best_train_mca'] = top_10_formatted['best_train_mca'].apply(lambda x: f"{x*100:.2f}%")
top_10_formatted['best_accuracy'] = top_10_formatted['best_accuracy'].apply(lambda x: f"{x*100:.2f}%")

sci_cols = ['config/lr', 'config/wd', 'config/eta_min']
for col in sci_cols:
    if col in top_10_formatted.columns:
        top_10_formatted[col] = top_10_formatted[col].apply(lambda x: f"{x:.2e}")

float_cols = [
    'config/dropout_p',
    'config/label_smoothing',
    'config/grad_clip_norm',
    'config/rgb_aug_prob',
    'config/rgb_aug_mag',
    'config/depth_aug_prob',
    'config/depth_aug_mag',
    'config/stem_lr_multiplier',
]
for col in float_cols:
    if col in top_10_formatted.columns:
        top_10_formatted[col] = top_10_formatted[col].apply(lambda x: f"{x:.3f}")

print(top_10_formatted.to_string(index=False))
print("\n" + "=" * 80)

In [ ]:
# =============================================================================
# ANALYZE TOP 10 TRIALS BY BEST VAL MCA - HYPERPARAMETER RANGES
# =============================================================================
import pandas as pd

df = results.get_dataframe()
df = df.sort_values("gap", ascending=True)
top_10 = df.head(10).copy()

config_cols = [c for c in df.columns if c.startswith("config/")]

print("=" * 80)
print("TOP 10 TRIALS BY BEST VAL MCA")
print("=" * 80)

for rank, (_, row) in enumerate(top_10.iterrows(), 1):
    gap = row.get("best_train_mca", 0) - row.get("best_val_mca", 0)
    print(f"\n--- #{rank} | Best Val MCA: {row['best_val_mca']*100:.2f}% | "
          f"Acc: {row['best_accuracy']*100:.2f}% | Gap: {gap*100:.1f}pp ---")

print("\n" + "=" * 80)
print("HYPERPARAMETER RANGES ACROSS TOP 10")
print("=" * 80)
print(f"{'Parameter':<35} {'Min':>12} {'Max':>12} {'Median':>12}")
print("-" * 75)

for col in config_cols:
    short_name = col.replace("config/", "")
    col_min = top_10[col].min()
    col_max = top_10[col].max()
    col_med = top_10[col].median()
    if abs(col_med) < 0.001:
        print(f"{short_name:<35} {col_min:>12.2e} {col_max:>12.2e} {col_med:>12.2e}")
    else:
        print(f"{short_name:<35} {col_min:>12.4f} {col_max:>12.4f} {col_med:>12.4f}")

In [ ]:
# =============================================================================
# FULL CONFIG TABLE - TOP 10 BY BEST VAL MCA
# =============================================================================
import pandas as pd

df = results.get_dataframe()
df = df.sort_values("best_val_mca", ascending=False)
top_10 = df.head(10).copy()

config_cols = [c for c in df.columns if c.startswith("config/")]
top_10["gap"] = top_10.get("best_train_mca", 0) - top_10.get("best_val_mca", 0)

display_df = top_10[["best_val_mca", "best_accuracy", "training_iteration"] + config_cols].copy()
display_df.insert(0, "rank", range(1, len(display_df) + 1))
display_df["best_val_mca"] = display_df["best_val_mca"].apply(lambda x: f"{x*100:.2f}%")
display_df["best_accuracy"] = display_df["best_accuracy"].apply(lambda x: f"{x*100:.2f}%")
display_df["training_iteration"] = display_df["training_iteration"].astype(int)

sci_cols = [c for c in config_cols if any(k in c for k in ["lr_", "wd_", "eta_min"]) and "multiplier" not in c]
float_cols = [c for c in config_cols if c not in sci_cols]

for col in sci_cols:
    if col in display_df.columns:
        display_df[col] = display_df[col].apply(lambda x: f"{x:.2e}")
for col in float_cols:
    if col in display_df.columns:
        display_df[col] = display_df[col].apply(lambda x: f"{x:.3f}")

display_df.columns = [c.replace("config/", "") for c in display_df.columns]

print(display_df.to_string(index=False))